# ML-Augmented Data Assimilation with the Lorenz-96 System
## A Summer School Tutorial

---

Welcome! This notebook is a guided introduction to **machine learning-augmented data assimilation (DA)**,
using the classic Lorenz-96 (L96) chaotic system as our testbed. By the end of this tutorial you will:

- Understand what data assimilation is and why it matters for Earth system prediction
- Be able to simulate the Lorenz-96 chaotic dynamical system
- Implement and run the **Ensemble Kalman Filter (EnKF)**
- Understand how a **convolutional neural network (CNN)** can be trained to replace or augment the EnKF analysis step
- Reproduce key results from the research paper this code accompanies
- Have a set of open-ended extension exercises to explore on your own

**Estimated time:** 2–3 hours (guided) + extension exercises

### Background paper
This code accompanies a study on combining machine learning with ensemble-based data assimilation.
The key idea: train a CNN to emulate the EnKF analysis step using data from a well-tuned EnKF run,
then alternate between the CNN and the EnKF in an "augmented" assimilation cycle.

### Repository structure
| File | Description |
|---|---|
| `L96.py` | Lorenz-96 model integration |
| `EnKF.py` | Ensemble Kalman Filter implementation |
| `Experiment.py` | High-level experiment orchestration |
| `NeuralNet.py` | 1-D CNN for learning the analysis step |
| `tutorial_notebook.ipynb` | **This notebook** |
| `generate_tutorial_data.py` | Script to pre-generate tutorial datasets |

## Prerequisites and Setup

**Python packages required:** `numpy`, `scipy`, `matplotlib`, `xarray`, `tqdm`, `keras`, `tensorflow`

Install with:
```bash
pip install numpy scipy matplotlib xarray tqdm tensorflow
```

**Background knowledge assumed:**
- Basic Python and NumPy
- Probability and statistics (mean, variance, Gaussian distributions)
- Some familiarity with neural networks (helpful but not required)

**Mathematical notation used in this notebook:**
- $\mathbf{x}$ — state vector (bold = vector)
- $\mathbf{x}^f$ — forecast (prior) estimate
- $\mathbf{x}^a$ — analysis (posterior) estimate
- $\mathbf{y}$ — observation vector
- $\mathbf{H}$ — observation operator (maps state space → observation space)
- $\mathbf{P}$ — forecast error covariance matrix
- $\mathbf{R}$ — observation error covariance matrix
- $\mathbf{K}$ — Kalman gain matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
from scipy.integrate import solve_ivp
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import the classes from this repository
from L96 import L96
from EnKF import EnKF
from Experiment import Experiment
from NeuralNet import NeuralNet

# Fix random seeds for reproducibility
np.random.seed(42)

# Plotting style
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})
print('All imports successful!')

---
# Part 1: The Lorenz-96 System

## 1.1 Background

The **Lorenz-96 (L96) model** was introduced by Edward Lorenz in 1996 as a minimal toy model of
large-scale atmospheric dynamics. Despite its simplicity it captures key atmospheric features:
chaotic behaviour, spatial structure, and energy cascades across scales.

L96 is the standard testbed for new data assimilation methods because:
1. It is cheap to run (a few seconds on a laptop)
2. It is genuinely chaotic — errors grow rapidly, just like in the real atmosphere
3. Its spatial structure (a periodic 1-D domain) mimics a latitude circle
4. We can generate perfect "synthetic" observations from the true trajectory

## 1.2 Mathematical Formulation

L96 describes $N$ variables $x_i$ ($i = 0, 1, \ldots, N-1$) arranged on a periodic ring:

$$
\frac{dx_i}{dt} = (x_{i+1} - x_{i-2})\, x_{i-1} - x_i + F
$$

where indices are taken modulo $N$ (cyclic boundary conditions), and:
- The $(x_{i+1} - x_{i-2}) x_{i-1}$ term represents nonlinear **advection** (energy transport)
- The $-x_i$ term represents linear **dissipation** (energy loss)
- $F$ is a constant **forcing** term (external energy input)

**Standard settings:** $N=40$, $F=8$. With $F=8$ the system is strongly chaotic with a
Lyapunov doubling time of about 2.1 time units (roughly corresponding to 2 days in atmospheric terms).

**Key time scales:**
- One L96 time unit $\approx$ 5 days in the real atmosphere
- We use $\Delta t = 0.05$ (about 6 hours) between observations

## 1.3 The L96 Class

Let's walk through the `L96` class from `L96.py`:

In [ ]:
# The L96 class is straightforward:
# L96(F, N) constructs the model with forcing F and N variables
# .derivative(t, x) computes dx/dt — this is what scipy's ODE solver calls
# .integrate(x0, t) integrates from initial state x0 over time array t

# Let's trace through the derivative manually for one variable:
N = 40
F = 8.0

# A test state at the fixed point (x_i = F for all i is an unstable equilibrium)
x_fp = F * np.ones(N)

# Perturb slightly to start off the fixed point (so chaos can develop)
x_fp[0] += 0.01

# Verify the derivative formula by hand for i=5
i = 5
d_manual = (x_fp[(i+1) % N] - x_fp[i-2]) * x_fp[i-1] - x_fp[i] + F
print(f'Derivative at i={i} (manual): {d_manual:.6f}')

# Now using the class
model = L96(F=F, N=N)
d_class = model.derivative(0, x_fp)
print(f'Derivative at i={i} (class) : {d_class[i]:.6f}')
print('Match:', np.isclose(d_manual, d_class[i]))

## 1.4 Generating a True Trajectory

In data assimilation, we call the actual system evolution the **"truth"**. In real applications
we never know the truth (that's why we do DA!). But in "identical twin" experiments with
toy models, we generate a synthetic truth and pretend we only get to observe it partially and noisily.

Let's integrate L96 to generate a truth trajectory and visualize it.

In [ ]:
# Parameters (matching the paper — we use a shorter T for the tutorial)
N = 40       # number of L96 variables
F = 8.0      # forcing (F=8 gives strong chaos)
dt = 0.05    # time step between observations (~6 hours)
T = 50.0     # total simulation time (~250 days)
            # The paper uses T=2000; we use T=50 for speed during the tutorial.
            # Change to T=200 for a longer, more statistically robust run.

# Initial condition: perturbed fixed point
x0 = F * np.ones(N)
x0[0] = F + 0.01  # small perturbation to break symmetry

# Time array
t_array = np.arange(0, T + dt, dt)
print(f'Simulating {len(t_array)} time steps over {T} time units')
print(f'That is approximately {T*5:.0f} days of atmosphere')

# Integrate
model = L96(F=F, N=N)
xx = model.integrate(x0, t_array)  # shape: (N, len(t_array))

print(f'\nTruth trajectory shape: {xx.shape}  (N x time)')
print(f'State range: [{xx.min():.2f}, {xx.max():.2f}]')
print(f'State std dev: {xx.std():.3f}')

In [ ]:
# ── Hovmöller diagram: x(space, time) ──────────────────────────────────────
# A Hovmöller diagram shows the state variable on the vertical axis,
# time on the horizontal axis — like a time-longitude plot in meteorology.

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot full domain
im = axes[0].pcolormesh(t_array, np.arange(N), xx, cmap='RdBu_r', vmin=-10, vmax=18)
axes[0].set_xlabel('Time (L96 units)')
axes[0].set_ylabel('Grid point index')
axes[0].set_title('L96 Truth: Hovmöller Diagram')
plt.colorbar(im, ax=axes[0], label='State value $x_i$')

# Time series at a single grid point
i_plot = 0
axes[1].plot(t_array, xx[i_plot, :], 'k-', linewidth=0.8)
axes[1].set_xlabel('Time (L96 units)')
axes[1].set_ylabel(f'$x_{{{i_plot}}}$')
axes[1].set_title(f'Time series at grid point {i_plot}')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_l96_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()
print('Notice the wave-like structures propagating in space and time!')

## 1.5 Demonstrating Chaos: Sensitivity to Initial Conditions

A key property of the L96 system is **sensitive dependence on initial conditions** — the hallmark
of chaos. Two trajectories starting from nearly identical initial states will diverge exponentially.
This is why weather prediction has a finite time horizon (about 2 weeks for the real atmosphere).

The rate of divergence is quantified by the **Lyapunov exponent** $\lambda$:
$$
\|\delta \mathbf{x}(t)\| \approx \|\delta \mathbf{x}(0)\|\, e^{\lambda t}
$$

For L96 with $F=8$, $\lambda \approx 0.47$ (doubling time $\approx$ 2.1 time units $\approx$ 10 days).

In [ ]:
# Two nearby initial conditions
eps = 1e-4  # tiny initial perturbation
x0_a = x0.copy()
x0_b = x0.copy()
x0_b[0] += eps  # perturb the first variable by epsilon

model = L96(F=F, N=N)
traj_a = model.integrate(x0_a, t_array)  # shape: (N, T)
traj_b = model.integrate(x0_b, t_array)

# Compute RMS distance between the two trajectories over time
diff = traj_b - traj_a
rms_dist = np.sqrt((diff**2).mean(axis=0))  # mean over spatial dimension

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Log-linear plot of divergence (straight line = exponential growth)
axes[0].semilogy(t_array, rms_dist, 'b-', linewidth=1.2, label='RMS distance')
# Overlay exponential growth reference line
lam_ref = 0.47  # approximate Lyapunov exponent
axes[0].semilogy(t_array, eps * np.exp(lam_ref * t_array), 'r--',
                 linewidth=1.5, label=f'$\epsilon \cdot e^{{\lambda t}}$, $\lambda={lam_ref}$')
axes[0].set_xlabel('Time (L96 units)')
axes[0].set_ylabel('RMS($x_b - x_a$)')
axes[0].set_title('Divergence of Two Nearby Trajectories')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Show the actual trajectories diverging at a single grid point
ax2 = axes[1]
ax2.plot(t_array, traj_a[0, :], 'b-', linewidth=0.9, label='Trajectory A')
ax2.plot(t_array, traj_b[0, :], 'r--', linewidth=0.9, label='Trajectory B')
ax2.set_xlabel('Time (L96 units)')
ax2.set_ylabel('$x_0$')
ax2.set_title('Two Trajectories at Grid Point 0')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_chaos.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Initial perturbation: {eps:.0e}')
saturation_time = t_array[np.argmax(rms_dist > rms_dist.max() * 0.5)]
print(f'Trajectories become decorrelated around t ≈ {saturation_time:.1f} time units')

## Exercise 1: Exploring the L96 System

The L96 system has interesting behaviour as a function of the forcing $F$:

| $F$ value | Behaviour |
|-----------|----------|
| $F < 5/9$ | Decays to zero (stable fixed point) |
| $F \approx 5/9$ | Periodic oscillations |
| $F = 5$ | Weakly chaotic |
| $F = 8$ | Strongly chaotic (our default) |
| $F = 16$ | Even more turbulent |

**Tasks:**
1. Modify the code above to run with $F=5$ and $F=16$. How does the Hovmöller diagram change?
2. Estimate the Lyapunov exponent for $F=5$ by fitting the slope in the log-linear divergence plot.
   Compare to $F=8$. Which system is more predictable?
3. **(Challenge)** The climatological variance of L96 depends on $F$. For $F=8$, the
   standard deviation is approximately 3.6. Confirm this empirically by running a long trajectory
   and computing the standard deviation of $x_i$ across all $i$ and all $t$.

In [ ]:
# ── Exercise 1 Workspace ──────────────────────────────────────────────────
# Your code here!

# Hint: to run with a different F, just change F in L96(F=..., N=N)
# and re-integrate. Try comparing Hovmöller diagrams side-by-side.

# Example scaffold:
for F_test in [5.0, 8.0, 16.0]:
    model_test = L96(F=F_test, N=N)
    traj_test = model_test.integrate(x0, t_array)
    print(f'F={F_test}: mean={traj_test.mean():.3f}, std={traj_test.std():.3f}')

---
# Part 2: Data Assimilation — Fusing Models and Observations

## 2.1 The Data Assimilation Problem

In geophysical applications we have two imperfect sources of information about the system state:

1. **A dynamical model** — gives a forecast $\mathbf{x}^f$ based on physics. Errors grow due to chaos and model imperfections.
2. **Observations** $\mathbf{y}$ — direct or indirect measurements of the system, available only at certain locations and times, corrupted by instrument noise.

**Data assimilation** is the principled statistical procedure for combining these two sources to
estimate the current system state. This analysis estimate $\mathbf{x}^a$ then serves as the
initial condition for the next forecast.

$$
\underbrace{\mathbf{x}^f}_{\text{model forecast}} \xrightarrow{\text{DA}} \underbrace{\mathbf{x}^a}_{\text{analysis}} \xrightarrow{\text{model}} \mathbf{x}^f_{t+\Delta t}
$$

## 2.2 The Bayesian Framework

DA is most naturally framed using **Bayes' theorem**:
$$
p(\mathbf{x} \mid \mathbf{y}) = \frac{p(\mathbf{y} \mid \mathbf{x})\, p(\mathbf{x})}{p(\mathbf{y})}
$$

- $p(\mathbf{x})$ — **prior** (our forecast, capturing model uncertainty)
- $p(\mathbf{y} \mid \mathbf{x})$ — **likelihood** (how well do observations match a given state?)
- $p(\mathbf{x} \mid \mathbf{y})$ — **posterior** (our analysis, after incorporating observations)

Under **Gaussian assumptions**, the optimal analysis is the **Best Linear Unbiased Estimator (BLUE)**:
$$
\mathbf{x}^a = \mathbf{x}^f + \mathbf{K}\left(\mathbf{y} - \mathbf{H}\mathbf{x}^f\right)
$$

where $\mathbf{K} = \mathbf{P}^f \mathbf{H}^T (\mathbf{H}\mathbf{P}^f\mathbf{H}^T + \mathbf{R})^{-1}$
is the **Kalman gain**.

The term $(\mathbf{y} - \mathbf{H}\mathbf{x}^f)$ is called the **innovation** or **departure** —
it measures how much the observations disagree with the forecast.

## 2.3 Observation Operators

The **observation operator** $\mathbf{H}$ maps from state space to observation space.
In our setup, observations are a subset of the L96 grid points — so $\mathbf{H}$ simply
selects certain indices from the state vector. The fraction of observed points is `frac`.

In [ ]:
# Reset to standard settings
F = 8.0
model = L96(F=F, N=N)

# Rebuild truth for standard params
x0 = F * np.ones(N)
x0[0] = F + 0.01
t_array = np.arange(0, T + dt, dt)
xx = model.integrate(x0, t_array)  # shape: (N, ntimes)
ntimes = xx.shape[1]

# ── Create synthetic observations ─────────────────────────────────────────
# Observation noise standard deviation
r = xx.std() * 0.3   # 30% of the climatological std dev (matching the paper)
print(f'State std dev: {xx.std():.3f}')
print(f'Observation noise std dev (r): {r:.3f}')
print(f'Observation noise as fraction of signal: {r/xx.std():.1%}')

# Add Gaussian noise to the truth to get observations
np.random.seed(42)
yy = xx + np.random.normal(0, r, xx.shape)

# For sparse observations: only observe a fraction of grid points
frac = 1.0  # fully observed (change to e.g. 0.5 for 50%)
obs_indices = list(range(N))  # all grid points observed

# ── Visualize: truth vs observations ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Pick a single grid point to visualize
i_show = 10
t_show = t_array[:100]  # show first 100 time steps

axes[0].plot(t_show, xx[i_show, :100], 'k-', linewidth=1.5, label='Truth')
axes[0].plot(t_show, yy[i_show, :100], 'r.', markersize=4, alpha=0.6, label='Observation')
axes[0].set_ylabel(f'$x_{{{i_show}}}$')
axes[0].set_title('Truth vs. Noisy Observations')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals (observation - truth)
resid = yy[i_show, :100] - xx[i_show, :100]
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].plot(t_show, resid, 'r.', markersize=3, alpha=0.7)
axes[1].axhline( r, color='gray', linestyle='--', label=f'±r = ±{r:.2f}')
axes[1].axhline(-r, color='gray', linestyle='--')
axes[1].set_xlabel('Time (L96 units)')
axes[1].set_ylabel('Obs − Truth')
axes[1].set_title('Observation Residuals')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_observations.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sparse observations: only observe a fraction of grid points ───────────
# In practice, observations are never complete. Let's look at what happens
# with 25% coverage (matching the augmented experiment in the paper).

frac_sparse = 0.25
nobs_sparse = int(N * frac_sparse)
obs_idx_sparse = np.random.choice(N, nobs_sparse, replace=False)
obs_idx_sparse.sort()

print(f'Fully observed:  {N} / {N} grid points ({100:.0f}%)')
print(f'Sparse observed: {nobs_sparse} / {N} grid points ({frac_sparse*100:.0f}%)')
print(f'Observed indices (sparse): {obs_idx_sparse}')

# Create sparse observation array (NaN at unobserved points)
yy_sparse = np.full_like(xx, np.nan)
yy_sparse[obs_idx_sparse, :] = xx[obs_idx_sparse, :] + np.random.normal(
    0, r, (nobs_sparse, ntimes)
)

# Show one snapshot
t_snap = 50  # snapshot index
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(np.arange(N), xx[:, t_snap], 'k-o', ms=5, label='Truth', linewidth=1.5)
ax.scatter(obs_idx_sparse, yy_sparse[obs_idx_sparse, t_snap],
           color='red', s=60, zorder=5, label='Sparse observations', marker='x')
ax.set_xlabel('Grid point index')
ax.set_ylabel('State value')
ax.set_title(f'Snapshot at t={t_array[t_snap]:.2f}: Truth vs 25% Sparse Observations')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('tutorial_sparse_obs.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercise 2: Observations and Their Properties

1. **Observation error distribution:** Plot a histogram of `yy - xx` across all grid points and
   times. What distribution do you expect? Does the empirical histogram match?
2. **Signal-to-noise ratio:** Compute the RMS of the truth and the RMS of the observation noise.
   How does the SNR change when you change `r` to `xx.std() * 0.1` (low noise) vs `xx.std() * 0.8` (high noise)?
3. **(Challenge)** The paper uses $r = 0.3 \times \sigma_{\text{clim}}$ for the base case.
   Look at `sensitivity.py` — what are the noise levels for cases 1 and 2?
   Why might different noise levels change the relative benefit of ML over the EnKF?

In [ ]:
# ── Exercise 2 Workspace ──────────────────────────────────────────────────
# Your code here!

# Hint for Q1: use plt.hist with 'density=True' and overlay a Gaussian curve
# Hint for Q2: np.sqrt(np.mean((yy - xx)**2)) gives observation RMSE

---
# Part 3: The Ensemble Kalman Filter

## 3.1 Why Ensembles?

The full Kalman filter requires maintaining and evolving the $N \times N$ error covariance matrix
$\mathbf{P}$ exactly. For L96 with $N=40$ this is only $40 \times 40$ — trivial!
But for a real atmospheric model with $N \sim 10^7$ or $10^9$ grid points, storing and evolving
$\mathbf{P}$ is computationally impossible.

The **Ensemble Kalman Filter (EnKF)** solves this by approximating $\mathbf{P}$ with a sample
covariance from a small ensemble of $K$ model runs:
$$
\mathbf{P}^f \approx \frac{1}{K-1} \sum_{k=1}^{K}\left(\mathbf{x}^f_k - \bar{\mathbf{x}}^f\right)\left(\mathbf{x}^f_k - \bar{\mathbf{x}}^f\right)^T
$$

This is the **stochastic EnKF** (Burgers, Evensen 1998): each ensemble member receives a
perturbed observation $\tilde{\mathbf{y}}_k = \mathbf{y} + \boldsymbol{\epsilon}_k$
where $\boldsymbol{\epsilon}_k \sim \mathcal{N}(0, \mathbf{R})$.

## 3.2 EnKF Algorithm

```
Initialize:  K ensemble members x_k ~ p(x_0)

For each observation time t_i:
    FORECAST STEP:
        For each ensemble member k:
            x^f_k(t_i) = M(x^a_k(t_{i-1}))   ← integrate model forward
    
    ANALYSIS STEP:
        Compute sample covariance:  P^f = sample_cov(x^f_1, ..., x^f_K)
        [Optional] Localize P^f    ← set far-away covariances to zero
        [Optional] Inflate P^f     ← multiply by gamma > 1
        Compute Kalman gain:  K = P^f H^T (H P^f H^T + R)^{-1}
        For each ensemble member k:
            y~_k = y + eps_k    (perturbed observation)
            x^a_k = x^f_k + K (y~_k - H x^f_k)
```

## 3.3 Covariance Localization

With a small ensemble ($K \ll N$), the sample covariance $\mathbf{P}^f$ has **spurious
long-range correlations** due to sampling error. **Localization** fixes this by
zeroing out covariance entries where the spatial distance exceeds a cutoff `loc`:

$$
P^f_{ij} = 0 \quad \text{if } d(i,j) > \text{loc}
$$

where $d(i,j)$ is the cyclic distance between grid points $i$ and $j$.

## 3.4 Covariance Inflation

Ensemble DA methods tend to **underestimate** forecast uncertainty over time
("filter divergence"). **Multiplicative inflation** multiplies the forecast covariance
by a factor $\gamma > 1$ to compensate:

$$
\mathbf{P}^f \leftarrow \gamma \, \mathbf{P}^f
$$

In [ ]:
# ── Walking through the EnKF class ────────────────────────────────────────

# The EnKF class is in EnKF.py. Let's trace through key methods:
#
# EnKF.obs(h, x)           -- apply observation operator H
# EnKF.getK(cov, h, r)     -- compute Kalman gain K
# EnKF.localize(cov)       -- apply distance-based localization
# EnKF.assimilate(...)     -- single analysis update
# EnKF.ensemble_assim(...) -- full ensemble analysis step

# Example: demonstrate Kalman gain calculation
nens = 20
loc = 5     # localization distance
gamma = 1.0 # no inflation
enkf_demo = EnKF(N=N, loc=loc, gamma=gamma, nens=nens)

# Create a fake forecast ensemble (20 members, each of length 40)
np.random.seed(0)
xf_demo = np.random.randn(N, nens) * 2 + 8  # rough L96-like spread

# Sample covariance from the ensemble
P_demo = np.cov(xf_demo)  # shape: (N, N)

# Show the raw (unlocalized) covariance structure
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(P_demo, cmap='RdBu_r', vmin=-3, vmax=3)
axes[0].set_title(f'Ensemble Covariance P (nens={nens}, unlocalized)')
axes[0].set_xlabel('Grid point j')
axes[0].set_ylabel('Grid point i')
plt.colorbar(im0, ax=axes[0])

# Apply localization
P_loc = enkf_demo.localize(P_demo.copy())
im1 = axes[1].imshow(P_loc, cmap='RdBu_r', vmin=-3, vmax=3)
axes[1].set_title(f'Localized Covariance P (loc={loc})')
axes[1].set_xlabel('Grid point j')
axes[1].set_ylabel('Grid point i')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig('tutorial_covariance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Notice how localization zeroes out long-range (spurious) covariances!')

In [ ]:
# ── Run a full EnKF experiment using the Experiment class ─────────────────
# The Experiment class orchestrates the full DA cycle:
#   - Stores truth, observations, forecast and analysis ensembles in an xarray.Dataset
#   - Handles the forecast-analysis cycling loop

np.random.seed(42)

# Settings (matching the paper's base case)
settings = {
    'N': 40,
    'F': 8.0,
    'dt': 0.05,
    'nens': 20,    # paper uses 100; we use 20 for speed during tutorial
    'loc': 5,
    'gamma': 1.0,
    'frac': 1.0,   # fully observed
}

T_run = 50.0   # total time (paper uses 2000; we use 50 for the tutorial)
               # Increase to T_run=200 or 500 for more robust statistics!

print('Setting up experiment...')
exp = Experiment(settings=settings)

# Generate the truth
x0_run = settings['F'] * np.ones(settings['N'])
x0_run[0] = settings['F'] + 0.01
exp.ds['xx'] = exp.get_true(x0_run, T_run)
print(f"Truth shape: {exp.ds['xx'].shape}")

# Set observation noise
exp.r = float(exp.ds.xx.std()) * 0.3
print(f'Observation noise r = {exp.r:.3f}')

# Create synthetic observations
exp.ds['yy'] = exp.makeobs(exp.r)
print(f"Observations shape: {exp.ds['yy'].shape}")

# Create initial ensemble (perturbed from truth at t=0)
xf0 = exp.make_ensemble(x0_run, exp.r)
print(f'Initial ensemble shape: {xf0.shape}')

print('\nRunning EnKF (this may take a minute)...')
xaens, xfens = exp.assimilate(xf0)
print(f'\nEnKF complete!')
print(f'Analysis ensemble shape: {xaens.shape}  (N x nens x time)')

In [ ]:
# ── Evaluate EnKF performance ─────────────────────────────────────────────

# Extract results from the experiment's dataset
xx_ds = exp.ds.xx  # truth, shape (N, T)
xa_mean = exp.ds.xaens.mean(dim='ensemble')  # ensemble-mean analysis
xf_mean = exp.ds.xfens.mean(dim='ensemble')  # ensemble-mean forecast
spread = exp.ds.xaens.std(dim='ensemble')     # ensemble spread

# Match time axis (assimilation starts at t=1 in the dataset)
xx_aligned = xx_ds.sel(time=xa_mean.time)

# RMSE: Root Mean Square Error (how far is the ensemble mean from truth?)
rmse_a = float(np.sqrt(((xa_mean - xx_aligned)**2).mean()))
rmse_f = float(np.sqrt(((xf_mean - xx_aligned)**2).mean()))
mean_spread = float(spread.mean())

print('=== EnKF Performance Summary ===')
print(f'Forecast RMSE:  {rmse_f:.4f}')
print(f'Analysis RMSE:  {rmse_a:.4f}')
print(f'Mean spread:    {mean_spread:.4f}')
print(f'Analysis improvement: {(rmse_f-rmse_a)/rmse_f*100:.1f}%')
print(f'\nClimatological RMSE (persistence):  {float(xx_aligned.std()):.4f}')
print(f'Spread/RMSE ratio: {mean_spread/rmse_a:.3f}  (ideal: 1.0 = well-calibrated ensemble)')

In [ ]:
# ── Visualize EnKF results ─────────────────────────────────────────────────

t_ds = xa_mean.time.values
i_show = 0

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ── Top panel: individual trajectories ────────────────────────────────────
ax = axes[0]
# Plot ensemble members (faint grey)
for k in range(min(settings['nens'], 10)):
    ax.plot(t_ds, exp.ds.xaens.isel(space=i_show, ensemble=k).values,
            'b-', alpha=0.15, linewidth=0.5)
# Truth
ax.plot(t_ds, xx_aligned.isel(space=i_show).values,
        'k-', linewidth=1.8, label='Truth')
# Ensemble mean analysis
ax.plot(t_ds, xa_mean.isel(space=i_show).values,
        'b-', linewidth=1.5, label='EnKF analysis (ensemble mean)')
ax.set_ylabel(f'$x_{{{i_show}}}$')
ax.set_title('EnKF Analysis vs. Truth')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

# ── Bottom panel: RMSE over time ───────────────────────────────────────────
ax2 = axes[1]
rmse_t = np.sqrt(((xa_mean - xx_aligned)**2).mean(dim='space'))  # RMSE over space
rmse_f_t = np.sqrt(((xf_mean - xx_aligned)**2).mean(dim='space'))
spread_t = spread.mean(dim='space')

ax2.plot(t_ds, rmse_f_t.values, 'r-', linewidth=1.2, alpha=0.7, label='Forecast RMSE')
ax2.plot(t_ds, rmse_t.values, 'b-', linewidth=1.2, label='Analysis RMSE')
ax2.plot(t_ds, spread_t.values, 'g--', linewidth=1.2, label='Ensemble spread')
ax2.set_xlabel('Time (L96 units)')
ax2.set_ylabel('RMSE / Spread')
ax2.set_title('EnKF Diagnostic Metrics Over Time')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_enkf_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Rank Histogram (Talagrand diagram) ───────────────────────────────────
# A rank histogram checks if the ensemble is statistically consistent with the truth.
# For each time step, we find the rank of the truth within the ensemble.
# A FLAT rank histogram means the ensemble is well-calibrated.
# - U-shaped: ensemble is overdispersive (too wide)
# - ∩-shaped: ensemble is underdispersive (too narrow) — common in practice!

def rank_histogram(truth, ensemble):
    """Compute rank histogram. truth: (N,T), ensemble: (N,K,T)"""
    N_s, K, T = ensemble.shape
    ranks = []
    for t in range(T):
        for n in range(N_s):
            # rank of truth among ensemble members
            rank = np.sum(ensemble[n, :, t] < truth[n, t])
            ranks.append(rank)
    return np.array(ranks)

# Get aligned arrays
xa_arr = exp.ds.xaens.values  # (N, nens, T)
xx_arr = xx_aligned.values    # (N, T)

ranks = rank_histogram(xx_arr, xa_arr)

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(settings['nens'] + 2) - 0.5
ax.hist(ranks, bins=bins, density=True, color='steelblue', edgecolor='white')
ax.axhline(1.0 / (settings['nens'] + 1), color='red', linestyle='--',
           label=f'Uniform (perfect: 1/{settings["nens"]+1})')
ax.set_xlabel('Rank of truth within ensemble')
ax.set_ylabel('Probability')
ax.set_title('Rank Histogram (Talagrand Diagram)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('tutorial_rank_histogram.png', dpi=120, bbox_inches='tight')
plt.show()
print('A flat histogram indicates a well-calibrated ensemble.')

## Exercise 3: EnKF Sensitivity

The EnKF has two key tuning parameters: **localization** (`loc`) and **inflation** (`gamma`).
The paper tests a range of these — that's what the `sensitivity.py` script does.

**Tasks:**
1. Re-run the EnKF experiment with different localization radii: `loc = 2, 5, 10, None`.
   How does the analysis RMSE change? What happens without any localization?
2. Try different inflation values: `gamma = 1.0, 1.01, 1.05, 1.1`.
   What is the effect? Is there an optimal value?
3. Re-run with `frac=0.25` (sparse observations, 25% observed). How does RMSE change?
   Does the optimal localization radius change with observation density?
4. **(Challenge)** Try reducing `nens` from 20 to 5. What happens to the analysis?
   How does this interact with localization? (Small ensembles need more aggressive localization!)

In [ ]:
# ── Exercise 3 Workspace ──────────────────────────────────────────────────
# Your code here! Below is a scaffold to run multiple EnKF configurations.

def run_enkf(N=40, F=8.0, dt=0.05, T=50.0, nens=20, loc=5, gamma=1.0, frac=1.0, seed=42):
    """Run an EnKF experiment and return analysis RMSE."""
    np.random.seed(seed)
    settings = {'N': N, 'F': F, 'dt': dt, 'nens': nens, 'loc': loc, 'gamma': gamma, 'frac': frac}
    exp = Experiment(settings=settings)
    x0 = F * np.ones(N)
    x0[0] = F + 0.01
    exp.ds['xx'] = exp.get_true(x0, T)
    exp.r = float(exp.ds.xx.std()) * 0.3
    exp.ds['yy'] = exp.makeobs(exp.r)
    xf0 = exp.make_ensemble(x0, exp.r)
    xaens, xfens = exp.assimilate(xf0)
    xa_mean = exp.ds.xaens.mean(dim='ensemble')
    xx_al = exp.ds.xx.sel(time=xa_mean.time)
    rmse = float(np.sqrt(((xa_mean - xx_al)**2).mean()))
    return rmse

# Example: compare localization values
# (This will take a few minutes — be patient!)
loc_values = [2, 5, 10]
for loc_test in loc_values:
    rmse_test = run_enkf(loc=loc_test)
    print(f'loc={loc_test:3d}: Analysis RMSE = {rmse_test:.4f}')

---
# Part 4: Machine Learning for Data Assimilation

## 4.1 Motivation

The EnKF works by learning the error statistics **online** from the current ensemble.
But what if we could instead **learn the analysis mapping offline**, training a neural
network to directly predict $\mathbf{x}^a$ from $(\mathbf{x}^f, \mathbf{y})$?

This is the approach taken in the augmented method:
- **Training data:** From a well-tuned EnKF run, save pairs of inputs and outputs:
  - Input: $(\mathbf{x}^f, \mathbf{y})$ — forecast and observation
  - Target: $\mathbf{x}^a$ — ensemble-mean analysis
- **Network:** A 1-D CNN that operates on the L96 ring (respecting its cyclic structure)
- **Inference:** For new $(\mathbf{x}^f, \mathbf{y})$, predict $\hat{\mathbf{x}}^a$ in one forward pass

**Why a CNN?** The L96 system has **translation invariance** — the same physics operates
at every grid point. A CNN exploits this by sharing weights across grid points, making
it much more data-efficient than a fully connected network.

## 4.2 The CNN Architecture

The network is a simple 3-layer 1-D CNN:

```
Input: (N + 2·offset, 2)   ← 2 channels: forecast x^f and innovation (y - x^f)
  │
  ├─ Conv1D(5 filters, kernel=3, relu)
  ├─ Conv1D(5 filters, kernel=3, relu)
  └─ Conv1D(1 filter,  kernel=3, linear)
  │
Output: (N, 1)             ← predicted analysis x^a
```

**Key design choices:**
- **Two input channels:** $\mathbf{x}^f$ and $(\mathbf{y} - \mathbf{x}^f)$ (the innovation)
  — feeding the innovation instead of $\mathbf{y}$ directly helps training
- **Cyclic padding:** L96 lives on a ring. The input is padded by wrapping around
  the edges so the CNN "sees" the correct neighbours at the boundaries.
- **Valid convolutions** (no boundary padding) ensure the output has exactly $N$ values

## 4.3 Cyclic Padding

The padding offset is computed from the architecture:
$$
\text{offset} = n_{\text{layers}} \times \frac{k-1}{2}
$$
where $k$ is the kernel size. For 3 layers with $k=3$: offset $= 3 \times 1 = 3$.

The input of size $N$ is extended to $N + 2 \times \text{offset}$ by wrapping:

In [ ]:
# ── Demonstrate cyclic padding ─────────────────────────────────────────────
# Let's trace through NeuralNet.make_input() to understand what it does.

nn_demo = NeuralNet(nlayers=3, filter_size=3, N=40)
offset = nn_demo.nlayers * (nn_demo.filter_size - 1) // 2
print(f'Number of layers: {nn_demo.nlayers}')
print(f'Filter size:      {nn_demo.filter_size}')
print(f'Cyclic padding offset: {offset}')
print(f'Input spatial dimension: {N} + 2×{offset} = {N + 2*offset}')

# Example with a single sample
xf_sample = np.random.randn(N) * 2 + 8   # fake forecast
y_sample  = xf_sample + np.random.randn(N) * 0.5  # fake observation

X_padded = nn_demo.make_input(xf_sample, y_sample)
print(f'\nInput to CNN shape: {X_padded.shape}  (1 sample × {N+2*offset} spatial × 2 channels)')

# Visualize the two channels
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
spatial_padded = np.arange(N + 2*offset)

axes[0].plot(spatial_padded, X_padded[0, :, 0], 'b-', linewidth=1.2)
axes[0].axvspan(0, offset, alpha=0.15, color='orange', label=f'Left padding ({offset} pts)')
axes[0].axvspan(N+offset, N+2*offset, alpha=0.15, color='orange', label=f'Right padding ({offset} pts)')
axes[0].set_ylabel('Channel 0: $x^f$')
axes[0].set_title('CNN Input: Cyclic Padding of Forecast')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(spatial_padded, X_padded[0, :, 1], 'r-', linewidth=1.2)
axes[1].axvspan(0, offset, alpha=0.15, color='orange')
axes[1].axvspan(N+offset, N+2*offset, alpha=0.15, color='orange')
axes[1].set_xlabel('Spatial index (padded)')
axes[1].set_ylabel('Channel 1: $y - x^f$ (innovation)')
axes[1].set_title('CNN Input: Cyclic Padding of Innovation')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_cyclic_padding.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Build and inspect the CNN ──────────────────────────────────────────────
nn = NeuralNet(nlayers=3, filter_size=3, N=40)
nn.buildmodel()
nn.model.summary()

## 4.4 Training the CNN

We use the EnKF run we already performed as training data:
- **Inputs:** $(\bar{\mathbf{x}}^f_t,\, \mathbf{y}_t)$ — ensemble-mean forecast and observations
- **Targets:** $\bar{\mathbf{x}}^a_t$ — ensemble-mean analysis from the EnKF

The network is trained to minimize the **mean squared error (MSE)** between its
prediction and the EnKF ensemble-mean analysis.

> **Note on the paper:** The full paper trains on a much longer run (T=2000 time steps)
> with 100 ensemble members. The best-performing sensitivity case (`s9`, loc=7)
> was used as training data. Here we train on our short tutorial run — the results
> will be suboptimal but the concepts are identical.

In [ ]:
# ── Train the CNN on the EnKF run ──────────────────────────────────────────
# ⚠ OVERFITTING WARNING ⚠
# With T=50 we generate ~700 training samples (70% of 1000 time steps).
# A 3-layer CNN has ~100 parameters, but the target function is complex, so
# the network will overfit: training RMSE drops but validation RMSE barely moves.
# This is EXPECTED and INTENTIONAL — it is a valuable teaching moment!
#
# To get meaningful generalisation:
#   • Increase T to 200–500 in the enkf-full-run cell and re-run
#   • Or load the pre-generated dataset:  ds = xr.open_dataset('tutorial_data.nc')
#     and construct Experiment.ds from it before calling nn.train()

import tensorflow as tf
tf.random.set_seed(42)

nn = NeuralNet(nlayers=3, filter_size=3, N=40)
nn.buildmodel()

# NeuralNet.train() expects the xarray Dataset (exp.ds), not the Experiment wrapper.
print('Training CNN...')
history = nn.train(
    training_fraction=0.7,  # 70% for training, 30% for validation
    nepochs=30,
    optimizer='adam',
    experiment=exp.ds        # pass the xarray Dataset (not the Experiment wrapper)
)
print('Training complete!')

train_final = history.history['root_mean_squared_error'][-1]
val_final   = history.history['val_root_mean_squared_error'][-1]
print(f'Final training RMSE:   {train_final:.4f}')
print(f'Final validation RMSE: {val_final:.4f}')
if val_final > 3 * train_final:
    print()
    print('>>> The large train/val gap confirms overfitting on this short run.')
    print('    Re-run with T=200+ for a well-generalising model.')

# Plot training history
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(history.history['root_mean_squared_error'], 'b-', label='Train RMSE')
ax.semilogy(history.history['val_root_mean_squared_error'], 'r--', label='Val RMSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE (log scale)')
ax.set_title('CNN Training Curve  (large gap = overfitting, expected with T=50)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('tutorial_training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Test the trained CNN on a single analysis step ─────────────────────────
# Pick a specific time step and compare EnKF vs CNN analysis

t_idx = 30  # which time step index to examine

# Get the time value at this index from the assimilation dataset
t_val = exp.ds.xaens.time.values[t_idx]

xf_test  = exp.ds.xfens.isel(time=t_idx).mean(dim='ensemble').values   # ensemble-mean forecast
y_test   = exp.ds.yy.sel(time=t_val).values                            # observation at that time
xa_enkf  = exp.ds.xaens.isel(time=t_idx).mean(dim='ensemble').values   # EnKF analysis
truth_t  = exp.ds.xx.sel(time=t_val).values                            # truth

# CNN prediction
xa_cnn = nn.assimilate(xf_test, y_test)

fig, ax = plt.subplots(figsize=(12, 4))
space = np.arange(N)
ax.plot(space, truth_t,  'k-o', ms=5,  linewidth=2,   label='Truth')
ax.plot(space, xf_test,  'g--',        linewidth=1.5, alpha=0.8, label='Forecast (ensemble mean)')
ax.plot(space, xa_enkf,  'b-',         linewidth=1.5, alpha=0.9, label='EnKF analysis')
ax.plot(space, xa_cnn,   'r-',         linewidth=1.5, alpha=0.9, label='CNN analysis')
ax.set_xlabel('Grid point index')
ax.set_ylabel('State value')
ax.set_title(f'Single-step comparison at time index {t_idx}  (t={t_val:.2f})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('tutorial_cnn_vs_enkf_snapshot.png', dpi=120, bbox_inches='tight')
plt.show()

rmse_f_snap    = np.sqrt(np.mean((xf_test - truth_t)**2))
rmse_enkf_snap = np.sqrt(np.mean((xa_enkf  - truth_t)**2))
rmse_cnn_snap  = np.sqrt(np.mean((xa_cnn   - truth_t)**2))
print(f'Snapshot RMSE — Forecast: {rmse_f_snap:.3f} | EnKF: {rmse_enkf_snap:.3f} | CNN: {rmse_cnn_snap:.3f}')

## Exercise 4: Exploring the Neural Network

1. **Architecture sensitivity:** Modify `NeuralNet.buildmodel()` in `NeuralNet.py` (or subclass it)
   to try a deeper network (4 or 5 layers) or wider filters (kernel size 5 or 7).
   Remember: the offset formula must be updated too! Does a more complex architecture help?
2. **Training data size:** Re-run with a longer EnKF simulation (`T=200` or `T=500`) to generate
   more training data. How does the validation RMSE change?
3. **Overfitting:** What does the training curve look like for 100 epochs? Add L2 regularization
   to the CNN layers and see if it helps: `Conv1D(..., kernel_regularizer='l2')`.
4. **(Challenge)** The CNN is trained on the ensemble-mean analysis. Would it be better to
   train it on the true state $\mathbf{x}^{\text{truth}}$ instead? Try it and compare.
   What are the trade-offs? (Hint: in real applications, we never know the truth!)

In [ ]:
# ── Exercise 4 Workspace ──────────────────────────────────────────────────
# Your code here!

# Hint: to modify the architecture, either:
#   (a) Edit NeuralNet.py directly, or
#   (b) Create a new Sequential model and assign it to nn.model directly:
#
# from keras import Sequential
# from keras.layers import Conv1D
# nn_custom = NeuralNet(nlayers=5, filter_size=3, N=40)
# nn_custom.buildmodel()  # then modify nn_custom.model
# ...
pass

---
# Part 5: The ML-Augmented Assimilation Method

## 5.1 The Augmented Algorithm

The key idea of the augmented method is to **alternate** between the CNN and EnKF:

```
For each observation time t_i:
    FORECAST STEP (same as EnKF)
    
    ANALYSIS STEP:
        if t_i is EVEN:  → use EnKF   (provides updated ensemble covariance)
        if t_i is ODD:   → use CNN    (fast, cheap; shifts the ensemble mean)
```

When the CNN is used, the ensemble spread is preserved by **shifting** all ensemble
members so their mean equals the CNN analysis:
$$
x^a_{k,i} = x^f_{k,i} - \left(\bar{x}^f_i - \hat{x}^a_i\right)
$$
where $\hat{x}^a_i$ is the CNN prediction. This keeps the ensemble from collapsing.

**Why does this work?**
The CNN uses the *full* observation vector $\mathbf{y}$ (not sparse), so even on
odd time steps where sparse observations are available, the CNN effectively
uses the information from the denser observations it was trained on.

This is most beneficial in **sparse observation** scenarios where the EnKF alone
is limited by the lack of observations.

## 5.2 Running the Augmented Experiment

In [ ]:
# ── Run the augmented (EnKF + CNN) experiment ─────────────────────────────
# We use sparse observations (frac=0.25) to test the augmented method,
# where the CNN's ability to "fill in" missing observations is most valuable.

np.random.seed(42)
tf.random.set_seed(42)

settings_aug = {
    'N': 40,
    'F': 8.0,
    'dt': 0.05,
    'nens': 20,
    'loc': 5,
    'gamma': 1.0,
    'frac': 0.25,  # SPARSE: only 25% of grid points observed
}

exp_aug_sparse = Experiment(settings=settings_aug)
exp_aug_sparse.ds['xx'] = exp.ds.xx   # reuse the same truth trajectory
exp_aug_sparse.r = float(exp_aug_sparse.ds.xx.std()) * 0.3
exp_aug_sparse.ds['yy'] = exp_aug_sparse.makeobs(exp_aug_sparse.r)

# Use a fresh initial ensemble (same perturbation as the full-obs run)
x0_aug = F * np.ones(N)
x0_aug[0] = F + 0.01
xf0_aug = exp_aug_sparse.make_ensemble(x0_aug, exp_aug_sparse.r)

print('Running augmented experiment (EnKF + CNN, sparse obs)...')
xaens_aug, xfens_aug = exp_aug_sparse.assimilate(xf0_aug, nn=nn)
print(f'Done!  Analysis ensemble shape: {xaens_aug.shape}')

In [ ]:
# ── Also run a pure EnKF with sparse observations for comparison ──────────
np.random.seed(42)

settings_sparse = {
    'N': 40, 'F': 8.0, 'dt': 0.05,
    'nens': 20, 'loc': 5, 'gamma': 1.0,
    'frac': 0.25,  # sparse observations
}

exp_sparse = Experiment(settings=settings_sparse)
exp_sparse.ds['xx'] = exp.ds.xx  # same truth
exp_sparse.r = float(exp_sparse.ds.xx.std()) * 0.3
exp_sparse.ds['yy'] = exp_sparse.makeobs(exp_sparse.r)
xf0_sparse = exp_sparse.make_ensemble(x0_aug, exp_sparse.r)

print('Running sparse EnKF...')
xaens_sp, xfens_sp = exp_sparse.assimilate(xf0_sparse, nn=None)
print('Done!')

In [ ]:
# ── Compare methods: EnKF (full obs) vs EnKF (sparse) vs Augmented (sparse) ─

def compute_rmse_timeseries(experiment):
    """Return spatial-mean RMSE time series for an experiment."""
    xa = experiment.ds.xaens.mean(dim='ensemble')
    xx = experiment.ds.xx.sel(time=xa.time)
    rmse_t = np.sqrt(((xa - xx)**2).mean(dim='space'))
    return rmse_t.values, xa.time.values

rmse_full,    t_full    = compute_rmse_timeseries(exp)
rmse_sparse,  t_sparse  = compute_rmse_timeseries(exp_sparse)
rmse_aug,     t_aug     = compute_rmse_timeseries(exp_aug_sparse)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Time series of RMSE ────────────────────────────────────────────────────
ax = axes[0]
ax.plot(t_full,   rmse_full,   'k-',  linewidth=1.2, label='EnKF (full obs)')
ax.plot(t_sparse, rmse_sparse, 'r-',  linewidth=1.2, alpha=0.8, label='EnKF (sparse 25%)')
ax.plot(t_aug,    rmse_aug,    'b--', linewidth=1.5, label='Augmented EnKF+CNN (sparse 25%)')
ax.set_xlabel('Time (L96 units)')
ax.set_ylabel('Analysis RMSE (spatial mean)')
ax.set_title('Analysis RMSE: EnKF vs. ML-Augmented Method')
ax.legend()
ax.grid(alpha=0.3)

# ── Bar chart of time-mean RMSE ────────────────────────────────────────────
ax2 = axes[1]
labels  = ['EnKF\n(full obs)', 'EnKF\n(sparse 25%)', 'Augmented\n(sparse 25%)']
means   = [rmse_full.mean(), rmse_sparse.mean(), rmse_aug.mean()]
colors  = ['black', 'red', 'blue']
bars = ax2.bar(labels, means, color=colors, alpha=0.7, edgecolor='white')
for bar, val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.002,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylabel('Time-mean Analysis RMSE')
ax2.set_title('Summary: Time-Mean RMSE by Method')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_method_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('=== Summary ===')
for label, rmse in zip(labels, means):
    print(f'{label.replace(chr(10)," "):30s}  RMSE = {rmse:.4f}')
improvement = (means[1] - means[2]) / means[1] * 100
print(f'\nAugmented improvement over sparse EnKF: {improvement:.1f}%')

## Exercise 5: Exploring the Augmented Method

1. **Alternation ratio:** The current code uses CNN every other step (odd steps).
   Modify `Experiment.assimilate()` to try:
   - CNN every step (replace EnKF entirely)
   - CNN every 4th step
   - EnKF every step (baseline)
   Which ratio gives the best performance? Why might using the CNN alone be problematic?

2. **Observation fraction:** Try the augmented method with different observation fractions
   (`frac = 0.5, 0.25, 0.1`). For which observation density does the ML augmentation
   help the most? Least?

3. **(Challenge)** In `Experiment.py` lines 191-196, the ensemble is adjusted so its
   mean equals the CNN analysis, but the **spread is unchanged**. Is this theoretically justified?
   What would happen if we also scaled the spread to better match the CNN's
   (presumably lower) error variance? Try modifying the code to rescale the spread.

4. **(Open research question)** The CNN was trained on a fully-observed experiment
   and then applied in a sparse observation setting. This is a form of **transfer**.
   Would a CNN trained specifically on sparse observations do better?
   What are the practical limitations of training on sparse-obs data?

In [ ]:
# ── Exercise 5 Workspace ──────────────────────────────────────────────────
# Your code here!

# Hint for Q1: look at Experiment.py lines 182-196.
# The condition `if i % 2 == 0` controls when EnKF vs CNN is used.
# You can change '% 2' to '% k' for different ratios.
# The cleanest approach is to subclass Experiment and override assimilate().
pass

---
# Part 6: Saving Results and Reproducing Paper Figures

## 6.1 Saving Experiment Results

The original paper saves experiments as pickle files and post-processes them separately.
Here we'll save our tutorial results as **netCDF files** using xarray — a more
interoperable format that works well with Python scientific tools.

In [ ]:
# ── Save tutorial results to netCDF ───────────────────────────────────────
# Save three experiments: full-obs EnKF, sparse EnKF, and augmented

# Save full-obs EnKF
exp.ds.to_netcdf('tutorial_enkf_full.nc')
print('Saved: tutorial_enkf_full.nc')

# Save sparse EnKF
exp_sparse.ds.to_netcdf('tutorial_enkf_sparse.nc')
print('Saved: tutorial_enkf_sparse.nc')

# Save augmented
exp_aug_sparse.ds.to_netcdf('tutorial_augmented.nc')
print('Saved: tutorial_augmented.nc')

# Save CNN weights
nn.model.save_weights('tutorial_cnn_weights.weights.h5')
print('Saved: tutorial_cnn_weights.weights.h5')

print('\nAll results saved!')

In [ ]:
# ── Load results and reproduce a summary figure ────────────────────────────
# This cell demonstrates how to load pre-saved results.
# (Useful if you want to run the analysis without re-running all experiments)

ds_full   = xr.open_dataset('tutorial_enkf_full.nc')
ds_sparse = xr.open_dataset('tutorial_enkf_sparse.nc')
ds_aug    = xr.open_dataset('tutorial_augmented.nc')

print('Loaded datasets:')
for name, ds in [('Full obs EnKF', ds_full), ('Sparse EnKF', ds_sparse), ('Augmented', ds_aug)]:
    print(f'  {name}: {dict(ds.dims)}')

# Quick sanity check: print time-mean RMSE for each
for name, ds in [('Full obs EnKF', ds_full), ('Sparse EnKF', ds_sparse), ('Augmented', ds_aug)]:
    if 'xaens' in ds:
        xa = ds.xaens.mean(dim='ensemble')
        xx = ds.xx.sel(time=xa.time)
        rmse_check = float(np.sqrt(((xa - xx)**2).mean()))
        print(f'  {name}: RMSE = {rmse_check:.4f}')

---
# Part 7: Extensions and Open Problems

Now that you've worked through the fundamentals, here are some directions to explore:

## Suggested Extensions (roughly in order of difficulty)

### Level 1: Reproduce the Paper
- **Run with paper settings:** Set `T=2000`, `nens=100`. This takes 20-60 minutes on a laptop.
  Can you reproduce the table of RMSE values in the paper?
- **Sensitivity analysis:** Run all 9 sensitivity cases from `sensitivity.py`.
  Which configuration is most predictable?

### Level 2: Method Improvements
- **Better inflation:** Try **adaptive inflation** — adjust $\gamma$ automatically based on
  the innovation statistics at each assimilation cycle.
- **Gaussian localization:** Replace the hard cutoff in `EnKF.localize()` with a smooth
  Gaspari-Cohn function. Does it improve results?
- **Deterministic EnKF:** Implement the Ensemble Transform Kalman Filter (ETKF) which
  avoids sampling noise from perturbed observations.

### Level 3: Different ML Approaches
- **LSTM/RNN:** Replace the CNN with a recurrent network that also uses information from
  previous time steps. Can it learn the temporal error structure?
- **Encoder-decoder:** Use a U-Net style architecture that might better capture multi-scale
  dependencies.
- **Variational autoencoder:** Train a VAE to represent the L96 manifold and do DA in
  the latent space.
- **Physics-informed:** Add a loss term that penalizes violations of the L96 equations.

### Level 4: Research Directions
- **Different test models:** Apply the augmented method to the 2-level Lorenz-96,
  or to a simple 2-D model.
- **Imperfect model:** Introduce model error by using a slightly wrong $F$ in the forecast
  model. Does the ML component help correct systematic model bias?
- **Observation timing:** What happens if the NN and EnKF use observations at different
  times (e.g., EnKF at $6h$, NN at $3h$)?
- **Explainability:** The paper uses SHAP values to understand which inputs the CNN focuses on.
  Look at `Plotting/postprocess_shap.py` — can you reproduce the SHAP analysis?

## Key Takeaways

1. **Data assimilation** is the problem of combining model forecasts with observations.
   The EnKF solves this using an ensemble of model trajectories.

2. **The EnKF requires tuning** (localization, inflation) and struggles with sparse observations.

3. **CNNs can learn the analysis mapping** from training data, exploiting the translation
   invariance of the L96 system.

4. **The augmented method** alternates between EnKF and CNN, with the CNN providing
   low-cost analyses in sparse observation scenarios.

5. **The L96 system is a useful testbed** but real applications require careful consideration
   of model complexity, observation operators, and generalization.

In [ ]:
# ── Final summary visualization ────────────────────────────────────────────
# Create a clean 4-panel summary figure

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('ML-Augmented Data Assimilation: Tutorial Summary', fontsize=14, fontweight='bold')

t_truth = t_array
i_show = 5  # grid point to show

# ── Panel 1: L96 truth (Hovmöller) ─────────────────────────────────────────
ax = axes[0, 0]
show_slice = slice(0, min(400, len(t_array)))
im = ax.pcolormesh(t_array[show_slice], np.arange(N),
                   xx[:, show_slice], cmap='RdBu_r', vmin=-10, vmax=18)
ax.set_xlabel('Time'); ax.set_ylabel('Grid point')
ax.set_title('(a) L96 Truth')
plt.colorbar(im, ax=ax)

# ── Panel 2: EnKF covariance (localized vs not) ─────────────────────────────
ax2 = axes[0, 1]
enkf_vis = EnKF(N=N, loc=5, gamma=1.0, nens=20)
np.random.seed(0)
xf_vis = np.random.randn(N, 20) * 2 + 8
P_vis = np.cov(xf_vis)
P_loc_vis = enkf_vis.localize(P_vis.copy())
im2 = ax2.imshow(P_loc_vis, cmap='RdBu_r', vmin=-2, vmax=2)
ax2.set_xlabel('Grid point j'); ax2.set_ylabel('Grid point i')
ax2.set_title('(b) Localized EnKF Covariance')
plt.colorbar(im2, ax=ax2)

# ── Panel 3: RMSE comparison (time series) ──────────────────────────────────
ax3 = axes[1, 0]
ax3.plot(t_full,   rmse_full,   'k-',  lw=1.2, label='EnKF (full obs)')
ax3.plot(t_sparse, rmse_sparse, 'r-',  lw=1.2, alpha=0.8, label='EnKF (sparse)')
ax3.plot(t_aug,    rmse_aug,    'b--', lw=1.5, label='Augmented (sparse)')
ax3.set_xlabel('Time (L96 units)'); ax3.set_ylabel('RMSE')
ax3.set_title('(c) Analysis RMSE Over Time')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── Panel 4: Training curve ─────────────────────────────────────────────────
ax4 = axes[1, 1]
ax4.semilogy(history.history['root_mean_squared_error'], 'b-', lw=1.2, label='Train')
ax4.semilogy(history.history['val_root_mean_squared_error'], 'r--', lw=1.2, label='Validation')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('RMSE (log scale)')
ax4.set_title('(d) CNN Training Curve')
ax4.legend(); ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tutorial_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Summary figure saved as tutorial_summary.png')

---
# Appendix: Quick Reference

## Key Parameters

| Parameter | Symbol | Typical value | Description |
|-----------|--------|---------------|-------------|
| `N` | $N$ | 40 | Number of L96 variables |
| `F` | $F$ | 8 | L96 forcing (controls chaos intensity) |
| `dt` | $\Delta t$ | 0.05 | Time step between observations |
| `nens` | $K$ | 100 | Ensemble size |
| `r` | $r$ | $0.3\sigma_{\text{clim}}$ | Observation noise std dev |
| `loc` | — | 5 | Localization radius (grid points) |
| `gamma` | $\gamma$ | 1.0–1.1 | Covariance inflation factor |
| `frac` | — | 0.25–1.0 | Fraction of grid points observed |

## Important Formulas

**L96 equation:**
$$\frac{dx_i}{dt} = (x_{i+1} - x_{i-2})\, x_{i-1} - x_i + F$$

**Kalman analysis:**
$$\mathbf{x}^a = \mathbf{x}^f + \mathbf{K}(\mathbf{y} - \mathbf{H}\mathbf{x}^f)$$

**Kalman gain:**
$$\mathbf{K} = \mathbf{P}^f \mathbf{H}^T (\mathbf{H}\mathbf{P}^f \mathbf{H}^T + \mathbf{R})^{-1}$$

**Analysis RMSE:**
$$\text{RMSE} = \sqrt{\frac{1}{N\,T}\sum_{i=1}^{N}\sum_{t=1}^{T} (x^a_{i,t} - x^{\text{truth}}_{i,t})^2}$$

## Useful Commands

```python
# Run a quick EnKF experiment
exp = Experiment(settings={'N':40,'F':8,'dt':0.05,'nens':20,'loc':5,'gamma':1,'frac':1})
exp.ds['xx'] = exp.get_true(x0, T)
exp.r = float(exp.ds.xx.std()) * 0.3
exp.ds['yy'] = exp.makeobs(exp.r)
xaens, xfens = exp.assimilate(exp.make_ensemble(x0, exp.r))

# Train CNN
nn = NeuralNet(nlayers=3, filter_size=3, N=40)
nn.buildmodel()
nn.train(training_fraction=0.7, nepochs=20, optimizer='adam', experiment=exp)

# Run augmented experiment
exp2.assimilate(xf0, nn=nn)   # pass nn to use the augmented method
```